# E1.4：RGG 空间邻近网络

张子瑜

本 notebook 展示一份 RGG 数据如何经过“生成数据 → 冻结模型预测 → ITE/ATE → 95% 区间与真值核对”。正式重复实验、覆盖率与基线比较见 `../results/e14_rgg/`。

节点位置位于单位正方形；距离不超过半径时连边。均匀分布和双簇聚集各用 0.12、0.18、0.24 三个半径。现有模型不接受孤立点，所以重抽整组位置直到没有孤立点；不补边，也不要求全图连通。空间位置只用于生成图和诊断，不进入预测器。

In [ ]:
from pathlib import Path
import sys
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'code/sparse_j').is_dir())
CODE = PROJECT/'code/sparse_j'
sys.path.insert(0, str(CODE)) if str(CODE) not in sys.path else None
import numpy as np
import pandas as pd
import torch
import networkx as nx
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
from benchmark import configuration, load_methods, predict_method
from benchmark_e14 import effects
from pfn_social.e14_rgg import SCENARIOS, generate_rgg
from pfn_social.phase6_limited_outcomes import make_observed_draw
from pfn_social.intervals import predict_intervals
from IPython.display import display

torch.set_num_threads(1)
limits = threadpool_limits(1)
[(s.setting_id, s.distribution, s.radius) for s in SCENARIOS]

## 生成一份数据

`SCENARIOS[0]` 为均匀分布、半径 0.12；可以换成其余五种场景。每份机制固定图、特征、系数与观测掩码，重复时只重抽处理和结果噪声。原有 7 类场景的代码和权重均保留。

In [ ]:
scenario = SCENARIOS[0]
mechanism, positions, graph_info = generate_rgg(scenario, mechanism_id=0)
rows, _ = make_observed_draw(mechanism, repeat_id=0)
row = next(r for r in rows if r['m_observed'] == 50)
inputs = {k: v.numpy().copy() for k, v in row['batch'].items()}
truth = mechanism.parent.truth.copy()  # 仅评测使用
print(graph_info)
print({k: v.shape for k, v in inputs.items()})
fig, ax = plt.subplots(figsize=(6, 5))
nx.draw_networkx_edges(nx.from_numpy_array(inputs['adjacency']), positions, ax=ax, alpha=.25, width=.5)
ax.scatter(*positions.T, s=12)
ax.set(xlim=(0,1), ylim=(0,1), aspect='equal', title='RGG: uniform, radius=0.12')
plt.show()
plt.close(fig)

## 调用模型与核对效应

四状态顺序是 `00,01,10,11`。直接效应为 10−00，已处理者溢出效应为 11−10，总效应为 11−00。ATE 对全部 200 个节点取平均。暴露状态内使用二项处理设计的条件平均，不能把低/高暴露简单替换成 0/1。

In [ ]:
model = load_methods(['j'], configuration())['j']
prediction = predict_method('j', model, inputs, seed=0)
interval = predict_intervals(model, **inputs, level=.95)
true_ite = effects(truth)
true_ate = true_ite.mean(0)
np.testing.assert_allclose(prediction['ite_effects'][:,2], prediction['ite_effects'][:,0]+prediction['ite_effects'][:,1])
summary = pd.DataFrame({
    'effect': ['direct','spillover','total'], 'true_ATE': true_ate,
    'estimated_ATE': prediction['ate_effects'],
    'lower95': interval['ate_lower'], 'upper95': interval['ate_upper'],
    'covered_this_draw': (interval['ate_lower'] <= true_ate) & (true_ate <= interval['ate_upper']),
    'ITE_RMSE': np.sqrt(np.mean((prediction['ite_effects']-true_ite)**2, axis=0))})
display(summary)
# 一份数据只能判断“此次是否覆盖”，不能估计长期覆盖率。

## 查看正式重复实验

下面只读取已完成的结果；正式实验由 `benchmark_e14.py` 生成，图表由原有 `make_figures.py`、`make_table.py` 与新增 `report_e14.py` 生成。覆盖率按独立机制聚类计算 Monte Carlo 标准误，不把相关节点当成独立重复。

In [ ]:
path = PROJECT/'results/e14_rgg/diagnostics/coverage_check.csv'
if path.exists():
    coverage = pd.read_csv(path)
    display(coverage)
    print('达到90%的配置：', int(coverage['meets_90_percent'].sum()), '/', len(coverage))
else:
    print('尚未生成正式结果；请先按 E14_README.md 运行命令。')